### Version 1

In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# --- Configuration & Parameters ---
TOTAL_USERS = 10000
START_DATE = datetime(2025, 1, 1)
END_DATE = datetime(2025, 9, 30)
OUTPUT_FILE = "ezTravel_demo_data_2025.csv"

# Realistic Destinations & Origins (Sampled from typical ezTravel data)
DESTINATIONS = ['Japan', 'Thailand', 'Taiwan', 'United States', 'Philippines', 'South Korea', 'Vietnam']
ORIGIN_COUNTRIES = ['Taiwan', 'Taiwan', 'Taiwan', 'Hong Kong', 'Macau'] # Weighted heavily to Taiwan
PRODUCT_TYPES = ['product_flight', 'product_hotel', 'product_trip', 'product_package']

# Generate the 10,000 Unique User IDs (Simulating ezTravel's 16-digit format)
user_ids = [str(random.randint(1000000000000000, 9999999999999999)) for _ in range(TOTAL_USERS)]

# Generate Date Range
date_range = [START_DATE + timedelta(days=x) for x in range((END_DATE - START_DATE).days + 1)]

# --- Trend Mapping (Peak vs Off-Season) ---
# We simulate the 4-5x peak ratio requested. 
# Base DAU = ~200. Medium = ~500. Peak = ~900 (Summer/Holidays).
def get_target_dau(current_date):
    month = current_date.month
    if month in [1, 2, 7, 8]: # High Peak (Lunar New Year, Summer Vacation)
        return int(np.random.normal(900, 100))
    elif month in [4, 5]: # Medium Peak (Spring Breaks)
        return int(np.random.normal(500, 50))
    else: # Off-season
        return int(np.random.normal(200, 30))

# --- Data Generation Loop ---
all_events = []
active_users_yesterday = set()
abandoned_carts = {} # Dictionary to track users who need a notification: {user_id: timestamp}

print("Generating complex multi-touchpoint data. This may take a minute...")

for current_date in date_range:
    target_dau = max(50, get_target_dau(current_date)) # Prevent negative DAU
    current_active_users = set()
    
    # 1. Retention Logic (Session Clusters)
    # 60% of people active yesterday are still researching today
    retained_users = set(random.sample(list(active_users_yesterday), int(len(active_users_yesterday) * 0.6)))
    current_active_users.update(retained_users)
    
    # Fill the rest with random users to hit the target DAU
    needed_users = max(0, target_dau - len(current_active_users))
    new_users = set(random.sample(user_ids, needed_users))
    current_active_users.update(new_users)
    
    # 2. Daily Event Funnel per User
    for uid in current_active_users:
        # Base timestamp for the user's session today
        base_ts = int(current_date.timestamp()) + random.randint(28800, 79200) # Between 8 AM and 10 PM
        current_ts = base_ts
        
        # User Properties for this session
        dest = random.choice(DESTINATIONS)
        origin = random.choice(ORIGIN_COUNTRIES)
        prod_id = f"PRD{random.randint(10000, 99999)}"
        
        current_ts = [base_ts]

        def add_event(name, time_offset_seconds):
            current_ts[0] += time_offset_seconds
            all_events.append({
                'date': current_date.strftime('%Y-%m-%d'),
                'event_name': name,
                'event_ts': current_ts[0],
                'qg_id': uid,
                'product_id': prod_id if 'product' in name or name in ['add_to_cart', 'booking_completed'] else 'null',
                'destination': dest,
                'origin_country': origin
            })

        # --- ORCHESTRATION: Re-engagement (Notification Sent) ---
        if uid in abandoned_carts:
            # Send notification 24 hours after abandonment
            add_event('notification_sent', 0)
            del abandoned_carts[uid] # Clear the cart flag
            
            # 40% chance the notification brings them back to book
            if random.random() < 0.40:
                add_event('visited', 120)
                add_event('product_viewed', 45)
                add_event('add_to_cart', 180)
                add_event('booking_completed', 300)
                continue # Session ends successfully

        # --- STANDARD FUNNEL: Browsing ---
        add_event('visited', random.randint(0, 10))
        add_event('page_viewed', random.randint(10, 30))
        
        # Intent: Do they view a product? (70% chance)
        if random.random() < 0.7:
            add_event('product_viewed', random.randint(20, 120))
            specific_product = random.choice(PRODUCT_TYPES)
            add_event(specific_product, random.randint(10, 60))
            
            # Action: Do they add to cart? (30% chance from product view)
            if random.random() < 0.3:
                add_event('add_to_cart', random.randint(30, 180))
                
                # Conversion: Do they book right away? (25% chance from add to cart)
                if random.random() < 0.25:
                    add_event('booking_completed', random.randint(120, 600))
                else:
                    # Abandoned Cart -> Queue for Notification tomorrow
                    abandoned_carts[uid] = current_ts

    # Update yesterday tracker
    active_users_yesterday = current_active_users

# --- DataFrame Creation and Output ---
df = pd.DataFrame(all_events)

# Sort strictly by timestamp to ensure chronological accuracy
df = df.sort_values(by='event_ts')

# Save to CSV
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
print(f"\nSuccess! Generated {len(df)} realistic travel events.")
print(f"Data saved to: {OUTPUT_FILE}")

# Print a quick preview of the funnel distribution
print("\n--- Event Distribution ---")
print(df['event_name'].value_counts())

Generating complex multi-touchpoint data. This may take a minute...

Success! Generated 594008 realistic travel events.
Data saved to: ezTravel_demo_data_2025.csv

--- Event Distribution ---
event_name
visited              155635
page_viewed          146957
product_viewed       111150
add_to_cart           39650
product_flight        25942
product_hotel         25530
product_package       25514
product_trip          25486
notification_sent     21718
booking_completed     16426
Name: count, dtype: int64


### Version 2

In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# --- Configuration & Parameters ---
TOTAL_USERS = 10000
START_DATE = datetime(2025, 1, 1)
END_DATE = datetime(2025, 9, 30)
OUTPUT_FILE = "HIS_Japan_demo_data_2025_new.csv"

# --- HIS-Specific Geographies ---
# Origin heavily weighted to major Japanese metropolitan areas
ORIGIN_CITIES = ['Tokyo', 'Tokyo', 'Tokyo', 'Osaka', 'Osaka', 'Nagoya', 'Fukuoka', 'Sapporo', 'Yokohama']

# Destinations: 80% Domestic, 20% Outbound (Typical for Japanese OTAs)
DOMESTIC_DESTS = ['Hokkaido (Sapporo)', 'Okinawa', 'Kyoto', 'Tokyo', 'Osaka', 'Fukuoka']
INTL_DESTS = ['Hawaii (USA)', 'South Korea', 'Taiwan', 'Thailand', 'Guam', 'Vietnam']
PRODUCT_TYPES = ['product_flight', 'product_hotel', 'product_trip', 'product_package']

# Generate the 10,000 Unique User IDs (Simulating standard 16-digit tracking IDs)
user_ids = [str(random.randint(1000000000000000, 9999999999999999)) for _ in range(TOTAL_USERS)]

# Generate Date Range
date_range = [START_DATE + timedelta(days=x) for x in range((END_DATE - START_DATE).days + 1)]

# --- HIS-Specific Trend Mapping (Japanese Holidays) ---
def get_target_dau(current_date):
    month = current_date.month
    day = current_date.day
    
    # 1. New Year (Shogatsu) - Late Dec to Jan 5
    if month == 1 and day <= 7:
        return int(np.random.normal(850, 100))
    # 2. Golden Week - Late April to Early May
    elif (month == 4 and day >= 25) or (month == 5 and day <= 7):
        return int(np.random.normal(950, 100)) # Absolute Peak
    # 3. Obon & Summer Break - August
    elif month == 8:
        if 10 <= day <= 20: # Core Obon week
            return int(np.random.normal(900, 100))
        return int(np.random.normal(600, 80))
    # 4. Spring Break / Sakura Season - Late March to Early April
    elif (month == 3 and day >= 20) or (month == 4 and day <= 10):
        return int(np.random.normal(550, 60))
    # 5. Silver Week - Mid September
    elif month == 9 and 15 <= day <= 25:
        return int(np.random.normal(500, 60))
    # Off-season (February, June, July, etc.)
    else:
        return int(np.random.normal(200, 30))

# --- Data Generation Loop ---
all_events = []
active_users_yesterday = set()
abandoned_carts = {} 

print("Generating HIS Japan multi-touchpoint data. This may take a minute...")

for current_date in date_range:
    target_dau = max(50, get_target_dau(current_date)) 
    current_active_users = set()
    
    # Session Clusters: 60% of people active yesterday are still researching today
    retained_users = set(random.sample(list(active_users_yesterday), int(len(active_users_yesterday) * 0.6)))
    current_active_users.update(retained_users)
    
    # Fill the rest with random users to hit the target DAU
    needed_users = max(0, target_dau - len(current_active_users))
    new_users = set(random.sample(user_ids, needed_users))
    current_active_users.update(new_users)
    
    # Daily Event Funnel per User
    for uid in current_active_users:
        base_ts = int(current_date.timestamp()) + random.randint(28800, 79200) 
        current_ts = base_ts
        
        # 80% Domestic, 20% International split (reflecting a 4:1 agency ratio)
        is_domestic = random.random() < 0.80
        dest = random.choice(DOMESTIC_DESTS) if is_domestic else random.choice(INTL_DESTS)
        origin_city = random.choice(ORIGIN_CITIES)
        
        # Prevent edge case of Tokyo to Tokyo flight by switching to Kyoto/Osaka
        if is_domestic and origin_city == dest:
            dest = 'Kyoto' if origin_city == 'Tokyo' else 'Tokyo'

        prod_id = f"HIS-PRD{random.randint(10000, 99999)}"
        
        current_ts = [base_ts]

        def add_event(name, time_offset_seconds):
            current_ts[0] += time_offset_seconds
            all_events.append({
                'date': current_date.strftime('%Y-%m-%d'),
                'event_name': name,
                'event_ts': current_ts[0],
                'qg_id': uid,
                'product_id': prod_id if 'product' in name or name in ['add_to_cart', 'booking_completed'] else 'null',
                'destination': dest,
                'origin_country': 'Japan',
                'origin_city': origin_city,
                'travel_type': 'Domestic' if is_domestic else 'Outbound'
            })

        # --- ORCHESTRATION: Re-engagement (Notification Sent) ---
        if uid in abandoned_carts:
            add_event('notification_sent', 0)
            del abandoned_carts[uid]
            
            if random.random() < 0.40:
                add_event('visited', 120)
                add_event('product_viewed', 45)
                add_event('add_to_cart', 180)
                add_event('booking_completed', 300)
                continue 

        # --- STANDARD FUNNEL: Browsing ---
        add_event('visited', random.randint(0, 10))
        add_event('page_viewed', random.randint(10, 30))
        
        if random.random() < 0.7:
            add_event('product_viewed', random.randint(20, 120))
            
            # Logic tweak: Packages are very popular for Hawaii/Okinawa
            if dest in ['Hawaii (USA)', 'Okinawa'] and random.random() < 0.6:
                specific_product = 'product_package'
            else:
                specific_product = random.choice(PRODUCT_TYPES)
                
            add_event(specific_product, random.randint(10, 60))
            
            if random.random() < 0.3:
                add_event('add_to_cart', random.randint(30, 180))
                
                if random.random() < 0.25:
                    add_event('booking_completed', random.randint(120, 600))
                else:
                    abandoned_carts[uid] = current_ts[0]

    active_users_yesterday = current_active_users

# --- DataFrame Creation and Output ---
df = pd.DataFrame(all_events)
df = df.sort_values(by='event_ts')

df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
print(f"\nSuccess! Generated {len(df)} realistic HIS travel events.")
print(f"Data saved to: {OUTPUT_FILE}")

Generating HIS Japan multi-touchpoint data. This may take a minute...

Success! Generated 371642 realistic HIS travel events.
Data saved to: HIS_Japan_demo_data_2025_new.csv


### Version 3

In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# --- Configuration & Parameters ---
TOTAL_USERS = 10000
START_DATE = datetime(2025, 1, 1)
END_DATE = datetime(2025, 9, 30)
OUTPUT_FILE = "HIS_Japan_Demo_2025_new.csv"

# --- HIS-Specific Geographies ---
ORIGIN_CITIES = ['Tokyo', 'Tokyo', 'Tokyo', 'Osaka', 'Osaka', 'Nagoya', 'Fukuoka', 'Sapporo', 'Yokohama']
DOMESTIC_DESTS = ['Hokkaido (Sapporo)', 'Okinawa', 'Kyoto', 'Tokyo', 'Osaka', 'Fukuoka']
INTL_DESTS = ['Hawaii (USA)', 'South Korea', 'Taiwan', 'Thailand', 'Guam', 'Vietnam']
PRODUCT_TYPES = ['product_flight', 'product_hotel', 'product_trip', 'product_package']

user_ids = [str(random.randint(1000000000000000, 9999999999999999)) for _ in range(TOTAL_USERS)]
date_range = [START_DATE + timedelta(days=x) for x in range((END_DATE - START_DATE).days + 1)]

# --- HIS-Specific Trend Mapping (Seasonality & Weekends) ---
def get_target_dau(current_date):
    month = current_date.month
    day = current_date.day
    is_weekend = current_date.weekday() >= 5 # 5 is Saturday, 6 is Sunday
    
    # Weekend multiplier (traffic generally increases on weekends for travel planning)
    weekend_mult = 1.35 if is_weekend else 1.0
    
    # 1. New Year (Shogatsu) - Late Dec to Jan 5
    if month == 1 and day <= 7:
        base_dau = np.random.normal(850, 100)
    # 2. Golden Week - Late April to Early May
    elif (month == 4 and day >= 25) or (month == 5 and day <= 7):
        base_dau = np.random.normal(950, 100) # Absolute Peak
    # 3. Obon & Summer Break - August
    elif month == 8:
        if 10 <= day <= 20: # Core Obon week
            base_dau = np.random.normal(900, 100)
        else:
            base_dau = np.random.normal(600, 80)
    # 4. Spring Break / Sakura Season - Late March to Early April
    elif (month == 3 and day >= 20) or (month == 4 and day <= 10):
        base_dau = np.random.normal(550, 60)
    # 5. Silver Week - Mid September
    elif month == 9 and 15 <= day <= 25:
        base_dau = np.random.normal(500, 60)
    # Off-season
    else:
        base_dau = np.random.normal(200, 30)
        
    return int(base_dau * weekend_mult)

# --- Dual-Peak Daily Distribution Logic (JST) ---
def get_realistic_timestamp(current_date):
    # Weights based on Japanese app usage: 
    # Morning commute (7-9 AM): 15%
    # Lunch break (12-2 PM): 25% (Peak 1)
    # Afternoon slack (2-6 PM): 15%
    # Evening / Pre-bed (7-11 PM): 45% (Peak 2 - Golden Hours)
    
    time_segment = random.choices(
        ['morning', 'lunch', 'afternoon', 'evening'],
        weights=[0.15, 0.25, 0.15, 0.45],
        k=1
    )[0]

    if time_segment == 'morning':
        hour = random.randint(7, 9)
    elif time_segment == 'lunch':
        hour = random.randint(12, 13)
    elif time_segment == 'afternoon':
        hour = random.randint(14, 18)
    else: # evening
        hour = random.randint(19, 23)

    minute = random.randint(0, 59)
    second = random.randint(0, 59)
    
    return int(current_date.replace(hour=hour, minute=minute, second=second).timestamp())

# --- Data Generation Loop ---
all_events = []
active_users_yesterday = set()
abandoned_carts = {} 

print("Generating highly granular HIS Japan data with Dual-Peak logic...")

for current_date in date_range:
    target_dau = max(50, get_target_dau(current_date)) 
    current_active_users = set()
    
    # Session Clusters
    retained_users = set(random.sample(list(active_users_yesterday), int(len(active_users_yesterday) * 0.6)))
    current_active_users.update(retained_users)
    
    needed_users = max(0, target_dau - len(current_active_users))
    new_users = set(random.sample(user_ids, needed_users))
    current_active_users.update(new_users)
    
    for uid in current_active_users:
        # Apply the new Dual-Peak Timestamp Logic
        current_ts = [get_realistic_timestamp(current_date)]
        
        # 80% Domestic, 20% International split (reflecting a 4:1 agency ratio)
        is_domestic = random.random() < 0.80
        dest = random.choice(DOMESTIC_DESTS) if is_domestic else random.choice(INTL_DESTS)
        origin_city = random.choice(ORIGIN_CITIES)
        
        if is_domestic and origin_city == dest:
            dest = 'Kyoto' if origin_city == 'Tokyo' else 'Tokyo'

        prod_id = f"HIS-PRD{random.randint(10000, 99999)}"

        def add_event(name, time_offset_seconds):
            current_ts[0] += time_offset_seconds
            all_events.append({
                'date': current_date.strftime('%Y-%m-%d'),
                'event_name': name,
                'event_ts': current_ts[0],
                'qg_id': uid,
                'product_id': prod_id if 'product' in name or name in ['add_to_cart', 'booking_completed'] else 'null',
                'destination': dest,
                'origin_country': 'Japan',
                'origin_city': origin_city,
                'travel_type': 'Domestic' if is_domestic else 'Outbound'
            })

        # Orchestration
        if uid in abandoned_carts:
            add_event('notification_sent', 0)
            del abandoned_carts[uid]
            
            if random.random() < 0.40:
                add_event('visited', 120)
                add_event('product_viewed', 45)
                add_event('add_to_cart', 180)
                add_event('booking_completed', 300)
                continue 

        # Standard Funnel
        add_event('visited', random.randint(0, 10))
        add_event('page_viewed', random.randint(10, 30))
        
        if random.random() < 0.7:
            add_event('product_viewed', random.randint(20, 120))
            
            if dest in ['Hawaii (USA)', 'Okinawa'] and random.random() < 0.6:
                specific_product = 'product_package'
            else:
                specific_product = random.choice(PRODUCT_TYPES)
                
            add_event(specific_product, random.randint(10, 60))
            
            if random.random() < 0.3:
                add_event('add_to_cart', random.randint(30, 180))
                
                if random.random() < 0.25:
                    add_event('booking_completed', random.randint(120, 600))
                else:
                    abandoned_carts[uid] = current_ts[0]

    active_users_yesterday = current_active_users

# --- DataFrame Creation and Output ---
df = pd.DataFrame(all_events)
df = df.sort_values(by='event_ts')

df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
print(f"\nSuccess! Generated {len(df)} realistic HIS travel events.")
print(f"Data saved to: {OUTPUT_FILE}")

Generating highly granular HIS Japan data with Dual-Peak logic...

Success! Generated 400728 realistic HIS travel events.
Data saved to: HIS_Japan_Demo_2025_new.csv


### Version 4: Country variation

In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# --- Configuration & Parameters ---
TOTAL_USERS = 10000
START_DATE = datetime(2025, 1, 1)
END_DATE = datetime(2025, 9, 30)
OUTPUT_FILE = "HIS_Demo_2025.csv"

# --- HIS-Specific Geographies ---
ORIGIN_CITIES = ['Tokyo', 'Tokyo', 'Tokyo', 'Osaka', 'Osaka', 'Nagoya', 'Fukuoka', 'Sapporo', 'Yokohama']
DOMESTIC_DESTS = ['Hokkaido (Sapporo)', 'Okinawa', 'Kyoto', 'Tokyo', 'Osaka', 'Fukuoka']

# International Destinations and their corresponding weights (must map 1:1)
INTL_DESTS =   ['South Korea', 'Hawaii (USA)', 'Taiwan', 'Thailand', 'Vietnam', 'Guam']
INTL_WEIGHTS = [0.30,          0.25,           0.20,     0.10,       0.10,      0.05]

PRODUCT_TYPES = ['product_flight', 'product_hotel', 'product_trip', 'product_package']

user_ids = [str(random.randint(1000000000000000, 9999999999999999)) for _ in range(TOTAL_USERS)]
date_range = [START_DATE + timedelta(days=x) for x in range((END_DATE - START_DATE).days + 1)]

# --- HIS-Specific Trend Mapping (Seasonality & Weekends) ---
def get_target_dau(current_date):
    month = current_date.month
    day = current_date.day
    is_weekend = current_date.weekday() >= 5 # 5 is Saturday, 6 is Sunday
    
    weekend_mult = 1.35 if is_weekend else 1.0
    
    # Seasonality Logic
    if month == 1 and day <= 7: # New Year
        base_dau = np.random.normal(850, 100)
    elif (month == 4 and day >= 25) or (month == 5 and day <= 7): # Golden Week
        base_dau = np.random.normal(950, 100)
    elif month == 8: # Obon
        if 10 <= day <= 20: 
            base_dau = np.random.normal(900, 100)
        else:
            base_dau = np.random.normal(600, 80)
    elif (month == 3 and day >= 20) or (month == 4 and day <= 10): # Sakura / Spring Break
        base_dau = np.random.normal(550, 60)
    elif month == 9 and 15 <= day <= 25: # Silver Week
        base_dau = np.random.normal(500, 60)
    else: # Off-season
        base_dau = np.random.normal(200, 30)
        
    return int(base_dau * weekend_mult)

# --- Dual-Peak Daily Distribution Logic (JST) ---
def get_realistic_timestamp(current_date):
    time_segment = random.choices(
        ['morning', 'lunch', 'afternoon', 'evening'],
        weights=[0.15, 0.25, 0.15, 0.45],
        k=1
    )[0]

    if time_segment == 'morning': hour = random.randint(7, 9)
    elif time_segment == 'lunch': hour = random.randint(12, 13)
    elif time_segment == 'afternoon': hour = random.randint(14, 18)
    else: hour = random.randint(19, 23)

    minute = random.randint(0, 59)
    second = random.randint(0, 59)
    
    return int(current_date.replace(hour=hour, minute=minute, second=second).timestamp())

# --- Data Generation Loop ---
all_events = []
active_users_yesterday = set()
abandoned_carts = {} 

print("Generating highly granular HIS Japan data with custom destination weights...")

for current_date in date_range:
    target_dau = max(50, get_target_dau(current_date)) 
    current_active_users = set()
    
    # Session Clusters
    retained_users = set(random.sample(list(active_users_yesterday), int(len(active_users_yesterday) * 0.6)))
    current_active_users.update(retained_users)
    
    needed_users = max(0, target_dau - len(current_active_users))
    new_users = set(random.sample(user_ids, needed_users))
    current_active_users.update(new_users)
    
    for uid in current_active_users:
        # Wrap timestamp in a list so inner function can mutate it
        current_ts = [get_realistic_timestamp(current_date)]
        
        # 80% Domestic, 20% International Split
        is_domestic = random.random() < 0.80
        
        # Apply the specific weights for International countries
        if is_domestic:
            dest = random.choice(DOMESTIC_DESTS)
        else:
            dest = random.choices(INTL_DESTS, weights=INTL_WEIGHTS, k=1)[0]
            
        origin_city = random.choice(ORIGIN_CITIES)
        
        # Prevent edge cases (e.g. Tokyo to Tokyo flight)
        if is_domestic and origin_city == dest:
            dest = 'Kyoto' if origin_city == 'Tokyo' else 'Tokyo'

        prod_id = f"HIS-PRD{random.randint(10000, 99999)}"
        
        def add_event(name, time_offset_seconds):
            current_ts[0] += time_offset_seconds
            all_events.append({
                'date': current_date.strftime('%Y-%m-%d'),
                'event_name': name,
                'event_ts': current_ts[0],
                'qg_id': uid,
                'product_id': prod_id if 'product' in name or name in ['add_to_cart', 'booking_completed'] else 'null',
                'destination': dest,
                'origin_country': 'Japan',
                'origin_city': origin_city,
                'travel_type': 'Domestic' if is_domestic else 'Outbound'
            })

        # Orchestration
        if uid in abandoned_carts:
            add_event('notification_sent', 0)
            del abandoned_carts[uid]
            
            if random.random() < 0.40:
                add_event('visited', 120)
                add_event('product_viewed', 45)
                add_event('add_to_cart', 180)
                add_event('booking_completed', 300)
                continue 

        # Standard Funnel
        add_event('visited', random.randint(0, 10))
        add_event('page_viewed', random.randint(10, 30))
        
        if random.random() < 0.7:
            add_event('product_viewed', random.randint(20, 120))
            
            # Hawaii and Okinawa searches highly skew towards Package products
            if dest in ['Hawaii (USA)', 'Okinawa'] and random.random() < 0.6:
                specific_product = 'product_package'
            else:
                specific_product = random.choice(PRODUCT_TYPES)
                
            add_event(specific_product, random.randint(10, 60))
            
            if random.random() < 0.3:
                add_event('add_to_cart', random.randint(30, 180))
                
                if random.random() < 0.25:
                    add_event('booking_completed', random.randint(120, 600))
                else:
                    # Store the latest timestamp for potential re-engagement
                    abandoned_carts[uid] = current_ts[0]

    active_users_yesterday = current_active_users

# --- DataFrame Creation and Output ---
df = pd.DataFrame(all_events)
df = df.sort_values(by='event_ts')

df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
print(f"\nSuccess! Generated {len(df)} realistic HIS travel events.")
print(f"Data saved to: {OUTPUT_FILE}")

Generating highly granular HIS Japan data with custom destination weights...

Success! Generated 402638 realistic HIS travel events.
Data saved to: HIS_Demo_2025.csv


### Version 5: Peak time during within a week

In [2]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# --- Configuration & Parameters ---
TOTAL_USERS = 10000
START_DATE = datetime(2025, 1, 1)
END_DATE = datetime(2025, 9, 30)
OUTPUT_FILE = "HIS_2025_demo_version.csv"

# --- HIS-Specific Geographies ---
ORIGIN_CITIES = ['Tokyo', 'Tokyo', 'Tokyo', 'Osaka', 'Osaka', 'Nagoya', 'Fukuoka', 'Sapporo', 'Yokohama']
DOMESTIC_DESTS = ['Hokkaido (Sapporo)', 'Okinawa', 'Kyoto', 'Tokyo', 'Osaka', 'Fukuoka']

# International Destinations and their corresponding weights (must map 1:1)
INTL_DESTS =   ['South Korea', 'Hawaii (USA)', 'Taiwan', 'Thailand', 'Vietnam', 'Guam']
INTL_WEIGHTS = [0.30,          0.25,           0.20,     0.10,       0.10,      0.05]

PRODUCT_TYPES = ['product_flight', 'product_hotel', 'product_package']

user_ids = [str(random.randint(1000000000000000, 9999999999999999)) for _ in range(TOTAL_USERS)]
date_range = [START_DATE + timedelta(days=x) for x in range((END_DATE - START_DATE).days + 1)]

# --- HIS-Specific Trend Mapping (Seasonality & Aviation Weekdays) ---
def get_target_dau(current_date):
    month = current_date.month
    day = current_date.day
    weekday = current_date.weekday()
    
    # NEW LOGIC: Aviation ticket buying peaks mid-week, drops on weekends
    if weekday == 2:          # Wednesday (Absolute Peak)
        day_mult = 1.35
    elif weekday in [1, 3]:   # Tuesday & Thursday (High traffic)
        day_mult = 1.15
    elif weekday >= 5:        # Saturday & Sunday (Lowest booking traffic)
        day_mult = 0.85
    else:                     # Monday & Friday (Baseline)
        day_mult = 1.0
    
    # Seasonality Logic
    if month == 1 and day <= 7: # New Year
        base_dau = np.random.normal(850, 100)
    elif (month == 4 and day >= 25) or (month == 5 and day <= 7): # Golden Week
        base_dau = np.random.normal(950, 100)
    elif month == 8: # Obon
        if 10 <= day <= 20: 
            base_dau = np.random.normal(900, 100)
        else:
            base_dau = np.random.normal(600, 80)
    elif (month == 3 and day >= 20) or (month == 4 and day <= 10): # Sakura / Spring Break
        base_dau = np.random.normal(550, 60)
    elif month == 9 and 15 <= day <= 25: # Silver Week
        base_dau = np.random.normal(500, 60)
    else: # Off-season
        base_dau = np.random.normal(200, 30)
        
    return int(base_dau * day_mult)

# --- Dual-Peak Daily Distribution Logic (JST) ---
def get_realistic_timestamp(current_date):
    time_segment = random.choices(
        ['morning', 'lunch', 'afternoon', 'evening'],
        weights=[0.15, 0.25, 0.15, 0.45],
        k=1
    )[0]

    if time_segment == 'morning': hour = random.randint(7, 9)
    elif time_segment == 'lunch': hour = random.randint(12, 13)
    elif time_segment == 'afternoon': hour = random.randint(14, 18)
    else: hour = random.randint(19, 23)

    minute = random.randint(0, 59)
    second = random.randint(0, 59)
    
    return int(current_date.replace(hour=hour, minute=minute, second=second).timestamp())

# --- Data Generation Loop ---
all_events = []
active_users_yesterday = set()
abandoned_carts = {} 

print("Generating highly granular HIS Japan data with custom destination weights...")

for current_date in date_range:
    target_dau = max(50, get_target_dau(current_date)) 
    current_active_users = set()
    
    # Session Clusters
    retained_users = set(random.sample(list(active_users_yesterday), int(len(active_users_yesterday) * 0.6)))
    current_active_users.update(retained_users)
    
    needed_users = max(0, target_dau - len(current_active_users))
    new_users = set(random.sample(user_ids, needed_users))
    current_active_users.update(new_users)
    
    d_str = current_date.strftime('%Y-%m-%d')
    
    for uid in current_active_users:
        # Standard integer assignment (No lists or nonlocal variables)
        current_ts = get_realistic_timestamp(current_date)
        
        # 80% Domestic, 20% International Split
        is_domestic = random.random() < 0.80
        
        if is_domestic:
            dest = random.choice(DOMESTIC_DESTS)
        else:
            dest = random.choices(INTL_DESTS, weights=INTL_WEIGHTS, k=1)[0]
            
        origin_city = random.choice(ORIGIN_CITIES)
        
        # Prevent edge cases (e.g. Tokyo to Tokyo flight)
        if is_domestic and origin_city == dest:
            dest = 'Kyoto' if origin_city == 'Tokyo' else 'Tokyo'

        prod_id = f"HIS-PRD{random.randint(10000, 99999)}"
        travel_type = 'Domestic' if is_domestic else 'Outbound'

        # Create a base dictionary template for this user's current session
        base_event = {
            'date': d_str,
            'qg_id': uid,
            'destination': dest,
            'origin_country': 'Japan',
            'origin_city': origin_city,
            'travel_type': travel_type
        }

        # Orchestration Logic
        if uid in abandoned_carts:
            all_events.append({**base_event, 'event_name': 'notification_sent', 'event_ts': current_ts, 'product_id': 'null'})
            del abandoned_carts[uid]
            
            if random.random() < 0.40:
                current_ts += 120
                all_events.append({**base_event, 'event_name': 'visited', 'event_ts': current_ts, 'product_id': 'null'})
                
                current_ts += 45
                all_events.append({**base_event, 'event_name': 'product_viewed', 'event_ts': current_ts, 'product_id': prod_id})
                
                current_ts += 180
                all_events.append({**base_event, 'event_name': 'add_to_cart', 'event_ts': current_ts, 'product_id': prod_id})
                
                current_ts += 300
                all_events.append({**base_event, 'event_name': 'booking_completed', 'event_ts': current_ts, 'product_id': prod_id})
                continue 

        # Standard Funnel Logic
        current_ts += random.randint(0, 10)
        all_events.append({**base_event, 'event_name': 'visited', 'event_ts': current_ts, 'product_id': 'null'})
        
        current_ts += random.randint(10, 30)
        all_events.append({**base_event, 'event_name': 'page_viewed', 'event_ts': current_ts, 'product_id': 'null'})
        
        if random.random() < 0.7:
            current_ts += random.randint(20, 120)
            all_events.append({**base_event, 'event_name': 'product_viewed', 'event_ts': current_ts, 'product_id': prod_id})
            
            # Hawaii and Okinawa searches highly skew towards Package products
            if dest in ['Hawaii (USA)', 'Okinawa'] and random.random() < 0.6:
                specific_product = 'product_package'
            else:
                specific_product = random.choice(PRODUCT_TYPES)
                
            current_ts += random.randint(10, 60)
            all_events.append({**base_event, 'event_name': specific_product, 'event_ts': current_ts, 'product_id': prod_id})
            
            if random.random() < 0.3:
                current_ts += random.randint(30, 180)
                all_events.append({**base_event, 'event_name': 'add_to_cart', 'event_ts': current_ts, 'product_id': prod_id})
                
                if random.random() < 0.25:
                    current_ts += random.randint(120, 600)
                    all_events.append({**base_event, 'event_name': 'booking_completed', 'event_ts': current_ts, 'product_id': prod_id})
                else:
                    # Store the latest timestamp for potential re-engagement
                    abandoned_carts[uid] = current_ts

    active_users_yesterday = current_active_users

# --- DataFrame Creation and Output ---
df = pd.DataFrame(all_events)
df = df.sort_values(by='event_ts')

df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
print(f"\nSuccess! Generated {len(df)} realistic HIS travel events.")
print(f"Data saved to: {OUTPUT_FILE}")

Generating highly granular HIS Japan data with custom destination weights...

Success! Generated 387567 realistic HIS travel events.
Data saved to: HIS_2025_demo_version.csv


### Version 6: More Detailed CSM

In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# --- Configuration & Parameters ---
TOTAL_USERS = 10000
START_DATE = datetime(2025, 1, 1)
END_DATE = datetime(2025, 9, 30)
OUTPUT_FILE = "HIS_2025_demo_version.csv"

# --- HIS-Specific Geographies ---
ORIGIN_CITIES = ['Tokyo', 'Tokyo', 'Tokyo', 'Osaka', 'Osaka', 'Nagoya', 'Fukuoka', 'Sapporo', 'Yokohama']
DOMESTIC_DESTS = ['Hokkaido (Sapporo)', 'Okinawa', 'Kyoto', 'Tokyo', 'Osaka', 'Fukuoka']

# International Destinations and their corresponding weights (must map 1:1)
INTL_DESTS =   ['South Korea', 'Hawaii (USA)', 'Taiwan', 'Thailand', 'Vietnam', 'Guam']
INTL_WEIGHTS = [0.30,          0.25,           0.20,     0.10,       0.10,      0.05]

PRODUCT_TYPES = ['product_flight', 'product_hotel', 'product_package']

user_ids = [str(random.randint(1000000000000000, 9999999999999999)) for _ in range(TOTAL_USERS)]
date_range = [START_DATE + timedelta(days=x) for x in range((END_DATE - START_DATE).days + 1)]

# --- HIS-Specific Trend Mapping (Seasonality & Aviation Weekdays) ---
def get_target_dau(current_date):
    month = current_date.month
    day = current_date.day
    weekday = current_date.weekday()
    
    # NEW LOGIC: Aviation ticket buying peaks mid-week, drops on weekends
    if weekday == 2:          # Wednesday (Absolute Peak)
        day_mult = 1.35
    elif weekday in [1, 3]:   # Tuesday & Thursday (High traffic)
        day_mult = 1.15
    elif weekday >= 5:        # Saturday & Sunday (Lowest booking traffic)
        day_mult = 0.85
    else:                     # Monday & Friday (Baseline)
        day_mult = 1.0
    
    # Seasonality Logic
    if month == 1 and day <= 7: # New Year
        base_dau = np.random.normal(850, 100)
    elif (month == 4 and day >= 25) or (month == 5 and day <= 7): # Golden Week
        base_dau = np.random.normal(950, 100)
    elif month == 8: # Obon
        if 10 <= day <= 20: 
            base_dau = np.random.normal(900, 100)
        else:
            base_dau = np.random.normal(600, 80)
    elif (month == 3 and day >= 20) or (month == 4 and day <= 10): # Sakura / Spring Break
        base_dau = np.random.normal(550, 60)
    elif month == 9 and 15 <= day <= 25: # Silver Week
        base_dau = np.random.normal(500, 60)
    else: # Off-season
        base_dau = np.random.normal(200, 30)
        
    return int(base_dau * day_mult)

# --- Dual-Peak Daily Distribution Logic (JST) ---
def get_realistic_timestamp(current_date):
    time_segment = random.choices(
        ['morning', 'lunch', 'afternoon', 'evening'],
        weights=[0.15, 0.25, 0.15, 0.45],
        k=1
    )[0]

    if time_segment == 'morning': hour = random.randint(7, 9)
    elif time_segment == 'lunch': hour = random.randint(12, 13)
    elif time_segment == 'afternoon': hour = random.randint(14, 18)
    else: hour = random.randint(19, 23)

    minute = random.randint(0, 59)
    second = random.randint(0, 59)
    
    return int(current_date.replace(hour=hour, minute=minute, second=second).timestamp())

# --- Data Generation Loop ---
all_events = []
active_users_yesterday = set()
abandoned_carts = {} 

print("Generating highly granular HIS Japan data with custom destination weights...")

for current_date in date_range:
    target_dau = max(50, get_target_dau(current_date)) 
    current_active_users = set()
    
    # Session Clusters
    retained_users = set(random.sample(list(active_users_yesterday), int(len(active_users_yesterday) * 0.6)))
    current_active_users.update(retained_users)
    
    needed_users = max(0, target_dau - len(current_active_users))
    new_users = set(random.sample(user_ids, needed_users))
    current_active_users.update(new_users)
    
    d_str = current_date.strftime('%Y-%m-%d')
    
    for uid in current_active_users:
        # Standard integer assignment (No lists or nonlocal variables)
        current_ts = get_realistic_timestamp(current_date)
        
        # 80% Domestic, 20% International Split
        is_domestic = random.random() < 0.80
        
        if is_domestic:
            dest = random.choice(DOMESTIC_DESTS)
        else:
            dest = random.choices(INTL_DESTS, weights=INTL_WEIGHTS, k=1)[0]
            
        origin_city = random.choice(ORIGIN_CITIES)
        
        # Prevent edge cases (e.g. Tokyo to Tokyo flight)
        if is_domestic and origin_city == dest:
            dest = 'Kyoto' if origin_city == 'Tokyo' else 'Tokyo'

        prod_id = f"HIS-PRD{random.randint(10000, 99999)}"
        travel_type = 'Domestic' if is_domestic else 'Outbound'

        # Create a base dictionary template for this user's current session
        base_event = {
            'date': d_str,
            'qg_id': uid,
            'destination': dest,
            'origin_country': 'Japan',
            'origin_city': origin_city,
            'travel_type': travel_type
        }

        # Orchestration Logic
        if uid in abandoned_carts:
            all_events.append({**base_event, 'event_name': 'notification_sent', 'event_ts': current_ts, 'product_id': 'null'})
            del abandoned_carts[uid]
            
            if random.random() < 0.40:
                current_ts += 120
                all_events.append({**base_event, 'event_name': 'visited', 'event_ts': current_ts, 'product_id': 'null'})
                
                current_ts += 45
                all_events.append({**base_event, 'event_name': 'product_viewed', 'event_ts': current_ts, 'product_id': prod_id})
                
                current_ts += 180
                all_events.append({**base_event, 'event_name': 'add_to_cart', 'event_ts': current_ts, 'product_id': prod_id})
                
                current_ts += 300
                all_events.append({**base_event, 'event_name': 'booking_completed', 'event_ts': current_ts, 'product_id': prod_id})
                continue 

        # Standard Funnel Logic
        current_ts += random.randint(0, 10)
        all_events.append({**base_event, 'event_name': 'visited', 'event_ts': current_ts, 'product_id': 'null'})
        
        current_ts += random.randint(10, 30)
        all_events.append({**base_event, 'event_name': 'page_viewed', 'event_ts': current_ts, 'product_id': 'null'})
        
        if random.random() < 0.7:
            current_ts += random.randint(20, 120)
            all_events.append({**base_event, 'event_name': 'product_viewed', 'event_ts': current_ts, 'product_id': prod_id})
            
            # Hawaii and Okinawa searches highly skew towards Package products
            if dest in ['Hawaii (USA)', 'Okinawa'] and random.random() < 0.6:
                specific_product = 'product_package'
            else:
                specific_product = random.choice(PRODUCT_TYPES)
                
            current_ts += random.randint(10, 60)
            all_events.append({**base_event, 'event_name': specific_product, 'event_ts': current_ts, 'product_id': prod_id})
            
            if random.random() < 0.3:
                current_ts += random.randint(30, 180)
                all_events.append({**base_event, 'event_name': 'add_to_cart', 'event_ts': current_ts, 'product_id': prod_id})
                
                if random.random() < 0.25:
                    current_ts += random.randint(120, 600)
                    all_events.append({**base_event, 'event_name': 'booking_completed', 'event_ts': current_ts, 'product_id': prod_id})
                else:
                    # Store the latest timestamp for potential re-engagement
                    abandoned_carts[uid] = current_ts

    active_users_yesterday = current_active_users

# --- DataFrame Creation and Output ---
df = pd.DataFrame(all_events)
df = df.sort_values(by='event_ts')

df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
print(f"\nSuccess! Generated {len(df)} realistic HIS travel events.")
print(f"Data saved to: {OUTPUT_FILE}")